# Scrap Serenity (@aleabitoreddit)

X(Twitter) 트윗 수집 → `analysis_Serenity.db`

- **post**: 일반 게시물
- **reply**: 답글
- **subscriber**: 구독자 전용 (`exclusivityInfo` 기반 감지)

## 1. Configuration

쿠키와 설정값을 여기서 수정하세요.

In [1]:
# ── Cookie ────────────────────────────────────────────────────────────────────
# X(Twitter) 쿠키 — 만료 시 브라우저에서 새 값 복사
COOKIES = {
    "auth_token": "19e6ebd36ae6b84003f240f23b5c01288dcd347b",
    "ct0": "c10611e28016d3ee3fa69ed884178c76d045603137f352eb58cd2bb6ca66a96c50fc4acf1afd4f1d8747aa7250f0199b51b178e1d589ed1d6cd6d6c95fb47fd9090756a17bca1f80105e5cfb858ad2a3",
    "twid": "u%3D1818125492823986176"
}

# ── Target ────────────────────────────────────────────────────────────────────
TARGET_USER = "aleabitoreddit"

# ── Options ───────────────────────────────────────────────────────────────────
TWEET_COUNT = None   # None = 전체, 숫자 = 최대 N개
DEBUG = False        # True = raw tweet._data JSON 덤프

# ── Paths (수정 불필요) ───────────────────────────────────────────────────────
import os
DB_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')),
    '..', '..', '..', '.claude', 'skills', 'MarketData', 'Personas', 'Serenity')

# 노트북 위치 기준 상대경로 → 절대경로로 고정
_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
DB_DIR = os.path.normpath(os.path.join(
    _notebook_dir, '..', 'Personas', 'Serenity'))
DB_PATH = os.path.join(DB_DIR, 'analysis_Serenity.db')

os.makedirs(DB_DIR, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/MarketData/Personas/Serenity/analysis_Serenity.db


## 2. Setup

In [2]:
import asyncio
import sqlite3
import json
import re
import tempfile
from datetime import datetime, timedelta, timezone
from twikit import Client

KST = timezone(timedelta(hours=9))
RATE_LIMIT_DELAY = 2
DUPLICATE_THRESHOLD = 10
MAX_RETRIES = 3


def to_kst(twitter_time_str):
    dt = datetime.strptime(twitter_time_str, "%a %b %d %H:%M:%S %z %Y")
    return dt.astimezone(KST).strftime("%Y-%m-%dT%H:%M:%S+09:00")


def extract_tickers(text):
    tickers = re.findall(r'\$([A-Za-z]+)', text)
    return list(dict.fromkeys([t.upper() for t in tickers]))


def get_full_content(tweet):
    note = (tweet._data
            .get('note_tweet', {})
            .get('note_tweet_results', {})
            .get('result', {}))
    if note and 'text' in note:
        return note['text']
    return tweet.full_text


def get_media_urls(tweet):
    urls = []
    if tweet.media:
        for m in tweet.media:
            if hasattr(m, 'media_url') and m.media_url:
                urls.append(m.media_url)
    return urls


def classify_tweet_type(tweet):
    if tweet._data.get('exclusivityInfo'):
        return 'subscriber'
    elif tweet.in_reply_to:
        return 'reply'
    else:
        return 'post'


def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS tweets (
            id TEXT PRIMARY KEY,
            user TEXT NOT NULL,
            type TEXT NOT NULL CHECK(type IN ('post', 'reply', 'subscriber')),
            created_at TEXT NOT NULL,
            content TEXT,
            tickers TEXT DEFAULT '[]',
            media TEXT DEFAULT '[]'
        )
    ''')
    conn.commit()
    return conn


def get_existing_ids(conn):
    cur = conn.execute('SELECT id FROM tweets')
    return set(row[0] for row in cur.fetchall())


def save_tweets(conn, tweets):
    saved = 0
    for tweet in tweets:
        content = get_full_content(tweet)
        tickers = json.dumps(extract_tickers(content), ensure_ascii=False)
        media = json.dumps(get_media_urls(tweet), ensure_ascii=False)
        tweet_type = classify_tweet_type(tweet)
        conn.execute(
            'INSERT OR IGNORE INTO tweets (id, user, type, created_at, content, tickers, media) VALUES (?,?,?,?,?,?,?)',
            (tweet.id, tweet.user.screen_name if tweet.user else TARGET_USER,
             tweet_type, to_kst(tweet.created_at), content, tickers, media))
        if conn.total_changes:
            saved += 1
    conn.commit()
    return saved


async def fetch_with_retry(coro_fn):
    for attempt in range(MAX_RETRIES):
        try:
            return await coro_fn()
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  ⚠ {e} — retry in {wait}s...")
            await asyncio.sleep(wait)


print("Setup complete.")

Setup complete.


## 3. Scrape

In [3]:
async def run_scrape():
    conn = init_db()
    existing_ids = get_existing_ids(conn)
    print(f"DB: {DB_PATH}")
    print(f"Existing: {len(existing_ids)}")

    # 쿠키를 임시 파일로 저장하여 twikit에 전달
    cookie_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
    json.dump(COOKIES, cookie_file)
    cookie_file.close()

    client = Client('en-US')
    try:
        client.load_cookies(cookie_file.name)
    except Exception as e:
        print(f"✗ Cookie expired — update COOKIES in cell 1.\n  {e}")
        conn.close()
        return
    finally:
        os.unlink(cookie_file.name)

    user = await client.get_user_by_screen_name(TARGET_USER)
    print(f"\n=== {user.name} (@{user.screen_name}) ===")
    print(f"Followers: {user.followers_count:,} | Tweets: {user.statuses_count:,}")

    total_saved = 0
    type_counts = {'post': 0, 'reply': 0, 'subscriber': 0}

    for tweet_type in ['Tweets', 'Replies']:
        print(f"\nFetching {tweet_type}...")
        consecutive_dups = 0
        total_fetched = 0

        try:
            tweets = await fetch_with_retry(
                lambda tt=tweet_type: user.get_tweets(tt, count=20))
        except Exception as e:
            print(f"  ⚠ Failed: {e}")
            continue

        while tweets:
            page_tweets = []
            for t in tweets:
                sn = t.user.screen_name if t.user else None
                if sn and sn.lower() != TARGET_USER.lower():
                    continue

                total_fetched += 1

                if DEBUG:
                    print(f"  [DEBUG] {t.id}: exclusivityInfo={t._data.get('exclusivityInfo')}, in_reply_to={t.in_reply_to}")

                if t.id in existing_ids:
                    consecutive_dups += 1
                    if consecutive_dups >= DUPLICATE_THRESHOLD:
                        print(f"  {DUPLICATE_THRESHOLD} consecutive dups — stopping")
                        break
                else:
                    consecutive_dups = 0
                    tt = classify_tweet_type(t)
                    type_counts[tt] += 1
                    page_tweets.append(t)
                    existing_ids.add(t.id)
                    if TWEET_COUNT and (total_saved + len(page_tweets)) >= TWEET_COUNT:
                        break

            if page_tweets:
                saved = save_tweets(conn, page_tweets)
                total_saved += saved
                print(f"  Fetched {total_fetched}, saved {total_saved} total")

            if consecutive_dups >= DUPLICATE_THRESHOLD:
                break
            if TWEET_COUNT and total_saved >= TWEET_COUNT:
                break

            await asyncio.sleep(RATE_LIMIT_DELAY)
            try:
                tweets = await fetch_with_retry(lambda: tweets.next())
                if not tweets:
                    break
            except Exception as e:
                print(f"  ⚠ Pagination stopped: {e}")
                break

        if TWEET_COUNT and total_saved >= TWEET_COUNT:
            break

    print(f"\n{'='*40}")
    print(f"Saved this run: {total_saved}")
    print(f"  post={type_counts['post']}, reply={type_counts['reply']}, subscriber={type_counts['subscriber']}")

    # DB 통계
    cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
    stats = cur.fetchall()
    total = sum(c for _, c in stats)
    print(f"\nTotal in DB: {total}")
    for t, c in stats:
        print(f"  {t}: {c}")
    conn.close()

await run_scrape()

DB: /Users/seongjin/Documents/Seongjin_Claude/Dumok-of-WallStreet/.claude/skills/MarketData/Personas/Serenity/analysis_Serenity.db
Existing: 621

=== Serenity (@aleabitoreddit) ===
Followers: 94,613 | Tweets: 3,608

Fetching Tweets...
  Fetched 19, saved 10 total
  10 consecutive dups — stopping

Fetching Replies...
  Fetched 13, saved 12 total
  10 consecutive dups — stopping

Saved this run: 12
  post=7, reply=2, subscriber=3

Total in DB: 633
  post: 544
  reply: 4
  subscriber: 85


## 4. Explore DB

In [4]:
# ── 최신 트윗 확인 ─────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    'SELECT id, type, created_at, substr(content,1,80), tickers FROM tweets ORDER BY created_at DESC LIMIT 10')
for row in cur:
    print(f"[{row[1]:10s}] {row[2]}  {row[3]}")
    if row[4] != '[]':
        print(f"             tickers: {row[4]}")
conn.close()

[post      ] 2026-03-04T23:53:49+09:00  Holy… How goated was this $EWY IV trade?

The South Korean Index volatility went
             tickers: ["EWY", "AAOI"]
[post      ] 2026-03-04T22:36:37+09:00  Well… this was probably the fastest 100%+ rally since $AXTI?

Feels like everyth
             tickers: ["AXTI", "AAOI", "LITE"]
[post      ] 2026-03-04T20:44:02+09:00  $GOOGL rejecting CPO is a massive structural win for the entire pluggable optics
             tickers: ["GOOGL", "LITE", "AAOI", "FN", "IQE", "AXTI"]
[subscriber] 2026-03-04T19:22:05+09:00  Looks like the sell America trade is over sooner than expected.

some thoughts, 
             tickers: ["MSFT", "RDDT", "HOOD"]
[post      ] 2026-03-04T17:08:42+09:00  If the Hormuz were closed, it looks like 80% of South Korea’s LNG imports would 
             tickers: ["EWY"]
[post      ] 2026-03-04T15:55:33+09:00  Feels like everyone is doom-posting KOSPI saying:

> “Look at the chart, it can’
             tickers: ["SNDK", "EWY", "TSM"

In [5]:
# ── 타입별 통계 ───────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
for t, c in cur:
    print(f"  {t}: {c}")
total = conn.execute('SELECT COUNT(*) FROM tweets').fetchone()[0]
print(f"  total: {total}")
conn.close()

  post: 544
  reply: 4
  subscriber: 85
  total: 633


In [6]:
# ── 구독자 전용 트윗 확인 ─────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT created_at, substr(content,1,100), tickers FROM tweets WHERE type='subscriber' ORDER BY created_at DESC LIMIT 10")
for row in cur:
    print(f"{row[0]}  {row[1]}")
    if row[2] != '[]':
        print(f"  tickers: {row[2]}")
    print()
conn.close()

2026-03-04T19:22:05+09:00  Looks like the sell America trade is over sooner than expected.

some thoughts, looks like a huge ca
  tickers: ["MSFT", "RDDT", "HOOD"]

2026-03-04T03:51:32+09:00  Lazy day today. Just ended up buying $INTC on the 5% drop, and $SE on the 18% drop just thought ER w
  tickers: ["INTC", "SE", "AAOI", "AXTI", "NBIS"]

2026-03-04T01:12:29+09:00  Ended up joining the CSP club for a little bit due to Iran conflict. 

All the option IV is too elev
  tickers: ["AXTI", "RDDT"]

2026-03-03T17:37:44+09:00  I wonder if landmark is an early indicator how photonics will do in US trading?

Last time it went u

2026-03-02T21:51:29+09:00  TLDR: Energy directed weapons largest beneficary.

I ended up long $LASR.

Stuff that may or may not
  tickers: ["LASR", "BKSY", "UMAC", "AIRO", "LPTH"]

2026-03-02T21:33:30+09:00  Not too many niche players, major winner was SpektreWorks and Anduril from what I've found. 

Big de
  tickers: ["NOC", "LMT", "BA", "RTX", "QUBT", "UMAC", "LASR"

In [7]:
# ── 특정 티커 검색 ────────────────────────────────────────────────────────
SEARCH_TICKER = "AXTI"  # ← 변경하여 검색

conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT type, created_at, substr(content,1,100) FROM tweets WHERE tickers LIKE ? ORDER BY created_at DESC",
    (f'%"{SEARCH_TICKER}"%',))
rows = cur.fetchall()
print(f"${SEARCH_TICKER} mentions: {len(rows)}")
for row in rows[:10]:
    print(f"  [{row[0]}] {row[1]}  {row[2]}")
conn.close()

$AXTI mentions: 73
  [post] 2026-03-04T22:36:37+09:00  Well… this was probably the fastest 100%+ rally since $AXTI?

Feels like everything around me keeps 
  [post] 2026-03-04T20:44:02+09:00  $GOOGL rejecting CPO is a massive structural win for the entire pluggable optics ecosystem.  

Most 
  [subscriber] 2026-03-04T03:51:32+09:00  Lazy day today. Just ended up buying $INTC on the 5% drop, and $SE on the 18% drop just thought ER w
  [subscriber] 2026-03-04T01:12:29+09:00  Ended up joining the CSP club for a little bit due to Iran conflict. 

All the option IV is too elev
  [post] 2026-03-03T21:32:25+09:00  I feel like Soitec ( $SLOIF ) is a free 2-3x if people have the patience to wait a year? 

They're a
  [post] 2026-03-03T06:21:01+09:00  Holy ****, this thesis was perhaps the most legendary one on X this year. 

$AXTI is now at $46.7 af
  [post] 2026-03-03T02:18:10+09:00  I happened to own every single top individual stock performer this year:

From $AXTI and $AAOI in ph
  [post] 2